# WAXAL ASR - Ensemble Submission

Combines predictions from **Whisper Small** and **Whisper Large-V3** using
heuristic selection per sample. Picks the better prediction based on quality signals.

**Prerequisites:** Run both `submission_zero_shot.csv` (small) and `submission_large_v3.csv` (large-v3) first.

## 1. Setup

In [ ]:
import csv, re, unicodedata
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path(".").resolve().parent
SUBMISSION_DIR = PROJECT_ROOT / "submissions"

small_path = SUBMISSION_DIR / "submission_zero_shot.csv"
large_path = SUBMISSION_DIR / "submission_large_v3.csv"
ensemble_path = SUBMISSION_DIR / "submission_ensemble.csv"
sample_path = PROJECT_ROOT / "SampleSubmission.csv"

assert small_path.exists(), f"Missing {small_path}"
assert large_path.exists(), f"Missing {large_path}"
print("Both submission files found.")

## 2. Load Predictions

In [ ]:
def load_submission(path):
    preds = {}
    with open(path, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            preds[row["ID"]] = row["Target"]
    return preds

small_preds = load_submission(small_path)
large_preds = load_submission(large_path)

all_ids = sorted(set(small_preds) | set(large_preds))
print(f"Small predictions: {len(small_preds)}")
print(f"Large predictions: {len(large_preds)}")
print(f"Total unique IDs:  {len(all_ids)}")

## 3. Quality Scoring Functions

In [ ]:
# Script ranges that indicate garbled output (Khmer, Devanagari, etc.)
GARBLED_SCRIPTS = {
    "KHMER", "DEVANAGARI", "THAI", "TIBETAN", "MYANMAR",
    "BENGALI", "GUJARATI", "KANNADA", "TAMIL", "TELUGU",
    "MALAYALAM", "SINHALA", "LAO", "GEORGIAN", "ARMENIAN",
    "ETHIOPIC", "CHEROKEE", "CANADIAN",
}

def is_garbled(text):
    """Check if text contains garbled non-Latin script characters."""
    if not text or len(text) <= 1:
        return True
    garbled_count = 0
    for ch in text:
        try:
            script = unicodedata.name(ch, "").split()[0]
            if script in GARBLED_SCRIPTS:
                garbled_count += 1
        except (ValueError, IndexError):
            pass
    return garbled_count / len(text) > 0.3

def is_repetitive(text):
    """Check if text is mostly one repeated word/phrase."""
    if not text:
        return True
    words = text.split()
    if len(words) <= 3:
        return False
    counts = Counter(words)
    most_common_count = counts.most_common(1)[0][1]
    return most_common_count > len(words) * 0.5

def is_empty(text):
    """Check if text is empty or trivially short."""
    return not text or len(text.strip()) <= 1

def quality_score(text):
    """Score a prediction: higher = better quality. Range 0-100."""
    if is_empty(text):
        return 0
    if is_garbled(text):
        return 5
    if is_repetitive(text):
        return 10
    
    score = 50
    # Prefer reasonable length (20-500 chars)
    length = len(text)
    if 20 <= length <= 500:
        score += 20
    elif 10 <= length <= 1000:
        score += 10
    
    # Prefer text with spaces (actual words, not garbled)
    word_count = len(text.split())
    if word_count >= 3:
        score += 10
    
    # Penalize high non-ASCII ratio (likely hallucinated diacritics)
    non_ascii = sum(1 for c in text if ord(c) > 127)
    if non_ascii / max(len(text), 1) > 0.5:
        score -= 15
    
    # Reward unique words (low repetition)
    unique_ratio = len(set(text.split())) / max(word_count, 1)
    score += int(unique_ratio * 20)
    
    return min(score, 100)

# Test the scoring
print("Score tests:")
print(f"  Empty string:  {quality_score('')}")
print(f"  Single dot:    {quality_score('.')}")
print(f"  Repetitive:    {quality_score('kwa kwa kwa kwa kwa kwa kwa kwa kwa kwa')}")
print(f"  Good sentence: {quality_score('Amaato abali gali ku mazzi amateefu')}")

## 4. Ensemble Selection

In [ ]:
# Read test IDs in order
test_ids = []
with open(PROJECT_ROOT / "Test.csv", "r", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        test_ids.append(row["ID"])

ensemble_preds = {}
stats = {"small_wins": 0, "large_wins": 0, "tie_large": 0}
lang_stats = {}

for tid in test_ids:
    lang = tid.split("_")[0]
    lang_stats.setdefault(lang, {"small": 0, "large": 0, "tie": 0})
    
    s_text = small_preds.get(tid, "")
    l_text = large_preds.get(tid, "")
    
    s_score = quality_score(s_text)
    l_score = quality_score(l_text)
    
    if s_score > l_score + 10:  # Small wins by clear margin
        ensemble_preds[tid] = s_text
        stats["small_wins"] += 1
        lang_stats[lang]["small"] += 1
    elif l_score >= s_score:  # Large wins or tie -> prefer large
        ensemble_preds[tid] = l_text
        if l_score > s_score:
            stats["large_wins"] += 1
            lang_stats[lang]["large"] += 1
        else:
            stats["tie_large"] += 1
            lang_stats[lang]["tie"] += 1
    else:
        ensemble_preds[tid] = l_text
        stats["large_wins"] += 1
        lang_stats[lang]["large"] += 1

print(f"Ensemble selection results:")
print(f"  Large-V3 wins:    {stats['large_wins']}")
print(f"  Small wins:       {stats['small_wins']}")
print(f"  Ties (use large): {stats['tie_large']}")
print()
for lang in sorted(lang_stats):
    ls = lang_stats[lang]
    total = ls['small'] + ls['large'] + ls['tie']
    print(f"  {lang}: small={ls['small']}, large={ls['large']}, tie={ls['tie']} (total={total})")

## 5. Write Ensemble Submission

In [ ]:
with open(ensemble_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ID", "Target"])
    for tid in test_ids:
        writer.writerow([tid, ensemble_preds.get(tid, "")])

print(f"Ensemble submission written to: {ensemble_path}")
print(f"Total predictions: {len(ensemble_preds)}")

# Validate
if sample_path.exists():
    with open(sample_path, "r", encoding="utf-8") as f:
        expected = {row["ID"] for row in csv.DictReader(f)}
    with open(ensemble_path, "r", encoding="utf-8") as f:
        submitted = {row["ID"] for row in csv.DictReader(f)}
    missing = expected - submitted
    empty = sum(1 for tid in test_ids if not ensemble_preds.get(tid, "").strip())
    if missing:
        print(f"WARNING: Missing {len(missing)} IDs!")
    elif empty:
        print(f"WARNING: {empty} IDs have empty transcriptions")
    else:
        print("Validation PASSED - all IDs present with transcriptions")

# Quick quality comparison
print("\n--- Quality comparison ---")
for label, preds in [("Small", small_preds), ("Large-V3", large_preds), ("Ensemble", ensemble_preds)]:
    vals = list(preds.values())
    empty_c = sum(1 for v in vals if not v.strip())
    garbled_c = sum(1 for v in vals if is_garbled(v))
    rep_c = sum(1 for v in vals if is_repetitive(v))
    avg_len = sum(len(v) for v in vals) / max(len(vals), 1)
    print(f"  {label:10s}: empty={empty_c}, garbled={garbled_c}, repetitive={rep_c}, avg_len={avg_len:.0f}")